<a href="https://colab.research.google.com/github/RegNLP/RePASs/blob/main/LinguisticScoreCalcuation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install textstat
!pip install wordfreq
!pip install hf_xet

In [ ]:
import torch
import numpy as np
import nltk
import textstat
from wordfreq import zipf_frequency
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from torch.nn.utils.rnn import pad_sequence
import json
import csv
import os
import time
from typing import List, Dict, Tuple, Optional, Union

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

class FluencyScore:
    """
    Evaluates text fluency using GPT-2's next-word prediction loss.

    Score Interpretation:
    - Scores range from 0.001 (worst) to 1.0 (best)
    - A score of 1.0 means the generated text is significantly more fluent than the source
    - A score of 0.5 means equal fluency to source
    - Scores below 0.5 indicate worse fluency than source
    - Scores are normalized relative to source text fluency

    Technical Details:
    - Uses 80-token chunks with padding when needed
    - Processes text through GPT-2 in evaluation mode
    - Loss is calculated using cross-entropy
    - Final score is normalized using: (1.3 + source_loss - generated_loss) / 1.3
    """
    def __init__(self, device=None, same_length=False):
        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model = GPT2LMHeadModel.from_pretrained("gpt2").to(self.device)
        self.tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
        if self.device.type == "cuda":
            self.model = self.model.half()
        self.model.eval()
        self.same_length = same_length
        self.max_output_length = 80

    def split_into_chunks(self, text: str) -> List[List[int]]:
        """Split text into token chunks of max_output_length"""
        if not text or not isinstance(text, str):
            return []

        tokens = self.tokenizer.encode(text)
        return [tokens[i:i + (self.max_output_length - 1)]
                for i in range(0, len(tokens), self.max_output_length - 1)]

    def preprocess_batch(self, decoded: List[str]) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor], int]:
        """Prepare input batches with padding and special tokens"""
        if not decoded or not isinstance(decoded, list):
            return None, None, 0

        all_chunks = []
        for dec in decoded:
            if not dec or not isinstance(dec, str):
                continue
            chunks = self.split_into_chunks(dec)
            if chunks:
                all_chunks.extend(chunks)

        if not all_chunks:
            return None, None, 0

        try:
            decs_inp = pad_sequence(
                [torch.LongTensor([self.tokenizer.bos_token_id] + chunk) for chunk in all_chunks],
                batch_first=True, padding_value=0
            ).to(self.device)

            decs_out = pad_sequence(
                [torch.LongTensor(chunk + [self.tokenizer.eos_token_id]) for chunk in all_chunks],
                batch_first=True, padding_value=-1
            ).to(self.device)

            return decs_inp, decs_out, len(all_chunks)
        except Exception as e:
            print(f"Error in batch preprocessing: {str(e)}")
            return None, None, 0

    def text2loss(self, text: List[str]) -> float:
        """Calculate average loss per token for given text"""
        if not text or not isinstance(text, list):
            return float('inf')

        txt_inp, txt_out, num_chunks = self.preprocess_batch(text)
        if num_chunks == 0 or txt_inp is None or txt_out is None:
            return float('inf')

        try:
            with torch.no_grad():
                model_outputs = self.model(input_ids=txt_inp)
                crit = torch.nn.CrossEntropyLoss(ignore_index=-1, reduction='none')
                loss = crit(model_outputs["logits"].view(-1, self.tokenizer.vocab_size),
                          txt_out.view(-1)).view(txt_out.shape)

                mask = (txt_inp != 0).float()
                non_pad_count = torch.sum(mask, dim=1)
                loss_per_chunk = torch.sum(loss, dim=1) / non_pad_count

            return loss_per_chunk.mean().item()
        except Exception as e:
            print(f"Error calculating text loss: {str(e)}")
            return float('inf')

    def score(self, sources: List[str], generateds: List[str], printing: bool = False) -> Dict:
        """
        Calculate fluency scores for generated texts relative to sources

        Args:
            sources: List of source texts
            generateds: List of corresponding generated texts
            printing: Whether to print scores

        Returns:
            Dictionary containing:
            - scores: Normalized fluency scores (higher is better)
            - sources_loss: Raw loss values for sources
            - generateds_loss: Raw loss values for generated texts
        """
        if not sources or not generateds or len(sources) != len(generateds):
            return {"scores": [], "sources_loss": [], "generateds_loss": []}

        sources_score = torch.tensor([self.text2loss([src]) for src in sources])
        generateds_score = torch.tensor([self.text2loss([gen]) for gen in generateds])

        scores = (1.3 + sources_score - generateds_score) / 1.3
        scores = torch.clamp(scores, 0.001, 1.0).tolist()

        if printing:
            print("[fluency]", scores)
        return {
            "scores": scores,
            "sources_loss": sources_score.tolist(),
            "generateds_loss": generateds_score.tolist()
        }

class ReadabilityScore:
    """
    Computes readability scores using Flesch-Kincaid Grade Level (FKGL) and
    Flesch Reading Ease (FRE) scores, then normalizes them to [0,1] range.

    Score Interpretation:
    - FKGL: Lower is better (easier to read)
    - FRE: Higher is better (easier to read)
    - Normalized scores:
      * FKGL: 1 - (score/18), clamped to [0,1]
      * FRE: score/100, clamped to [0,1]
    - Combined score is average of normalized FKGL and FRE

    Typical Ranges:
    - FKGL: 0 (easiest) to 18 (hardest)
    - FRE: 0 (hardest) to 100 (easiest)
    """
    def compute(self, passage: str, answer: str) -> Dict:
        """
        Compute readability metrics for both passage and answer

        Args:
            passage: Source text
            answer: Generated text

        Returns:
            Dictionary containing:
            - fkgl: Tuple of (passage_score, answer_score)
            - fre: Tuple of (passage_score, answer_score)
            - readability_score: Tuple of normalized combined scores
        """
        if not passage or not answer:
            return {
                "fkgl": (0, 0),
                "fre": (0, 0),
                "readability_score": (0, 0)
            }

        try:
            fkgl_p = textstat.flesch_kincaid_grade(passage)
            fre_p = textstat.flesch_reading_ease(passage)
            fkgl_a = textstat.flesch_kincaid_grade(answer)
            fre_a = textstat.flesch_reading_ease(answer)

            norm_fkgl_p = max(0.0, min(1.0, 1 - fkgl_p / 18))
            norm_fkgl_a = max(0.0, min(1.0, 1 - fkgl_a / 18))
            norm_fre_p = max(0.0, min(1.0, fre_p / 100))
            norm_fre_a = max(0.0, min(1.0, fre_a / 100))

            combined_p = round((norm_fkgl_p + norm_fre_p) / 2, 5)
            combined_a = round((norm_fkgl_a + norm_fre_a) / 2, 5)

            return {
                "fkgl": (fkgl_p, fkgl_a),
                "fre": (fre_p, fre_a),
                "readability_score": (combined_p, combined_a)
            }
        except Exception as e:
            print(f"Error calculating readability: {str(e)}")
            return {
                "fkgl": (0, 0),
                "fre": (0, 0),
                "readability_score": (0, 0)
            }

def shift_to_score(shift: float, target_shift: float, right_slope: float = 0.25) -> float:
    """
    Convert a measured shift to a normalized score [0,1]

    Score Interpretation:
    - For shifts <= target: Linear increase from 0 to 1
    - For shifts > target: Linear decrease with specified slope
    - Always clamped to [0,1] range

    Visual Representation:

    Score
    1.0 |     /\
        |    /  \
        |   /    \
        |  /      \
    0.0 +-----------> Shift
        0   target
    """
    if shift <= target_shift:
        score = shift / (target_shift + 0.001)
    else:
        score = 1.0 - right_slope * (shift - target_shift) / (target_shift + 0.001)
    return np.clip(score, 0, 1.0)

class SimplicityScore:
    """
    Evaluates text simplicity through lexical and syntactic features.

    Components:
    1. Lexical Simplicity:
       - Uses Zipf word frequency (higher frequency = simpler)
       - Measures shift from source to generated text
       - Applies shift_to_score normalization

    2. Syntactic Simplicity:
       - Uses Flesch-Kincaid Grade Level (FKGL)
       - Measures reduction in grade level (shift)
       - Dynamic target shift based on source complexity

    Score Interpretation:
    - Both components normalized to [0,1] range
    - Final score is average of lexical and syntactic scores
    - Higher scores indicate greater simplicity improvement
    """
    def __init__(self, max_grade: int = 30, target_lexical_shift: float = 0.4):
        self.max_grade = max_grade
        self.target_lexical_shift = target_lexical_shift
        self.stopws = set(nltk.corpus.stopwords.words("english") + ["might", "would", "``"])

    def is_good_word(self, w: str) -> bool:
        """Filter out stopwords, numbers, and punctuation"""
        return (
            len(w) > 1 and len(w) < 30 and
            "'" not in w and
            not all(c.isdigit() for c in w) and
            w.lower() not in self.stopws
        )

    def word_score_func(self, w: str) -> float:
        """Get Zipf frequency for a word (higher = more common)"""
        return zipf_frequency(w, 'en', wordlist="large")

    def compute_lexical_zipf(self, text: str) -> float:
        """Calculate average Zipf frequency for content words"""
        if not text:
            return 0.0

        try:
            words = nltk.tokenize.word_tokenize(text)
            good_words = [w.lower() for w in words if self.is_good_word(w)]
            if not good_words:
                return 0.0
            return np.mean([self.word_score_func(w) for w in good_words])
        except:
            return 0.0

    def lexical_score(self, passage: str, answer: str) -> Tuple[float, float, float, float]:
        """
        Calculate lexical simplicity score based on Zipf frequency shift

        Returns:
            (source_zipf, generated_zipf, shift_amount, normalized_score)
        """
        zipf_source = self.compute_lexical_zipf(passage)
        zipf_generated = self.compute_lexical_zipf(answer)
        shift = zipf_generated - zipf_source
        score = shift_to_score(shift, target_shift=self.target_lexical_shift)
        return zipf_source, zipf_generated, shift, score

    def syntactic_score(self, passage: str, answer: str) -> Tuple[float, float, float, float]:
        """
        Calculate syntactic simplicity score based on FKGL reduction

        Returns:
            (source_fkgl, generated_fkgl, shift_amount, normalized_score)
        """
        try:
            fkgl_source = textstat.flesch_kincaid_grade(passage)
            fkgl_generated = textstat.flesch_kincaid_grade(answer)
            rshift = fkgl_source - fkgl_generated

            # Dynamic target shift based on source complexity
            if fkgl_source <= 4.0:
                target_shift = 0  # Already very simple
            elif fkgl_source <= 12.0:
                target_shift = (fkgl_source - 3) * 0.5
            else:
                target_shift = 4.5 + (fkgl_source - 12) * 0.83

            score = shift_to_score(rshift, target_shift=target_shift)
            return fkgl_source, fkgl_generated, rshift, score
        except:
            return 0, 0, 0, 0

    def compute(self, passage: str, answer: str) -> Dict:
        """
        Compute complete simplicity metrics

        Returns:
            Dictionary containing all lexical and syntactic metrics plus combined score
        """
        if not passage or not answer:
            return {
                "source_zipf": 0,
                "generated_zipf": 0,
                "zipf_shift": 0,
                "lexical_score": 0,
                "source_fkgl": 0,
                "generated_fkgl": 0,
                "fkgl_shift": 0,
                "syntactic_score": 0,
                "simplicity_score": 0
            }

        zipf_source, zipf_gen, zipf_shift, lex_score = self.lexical_score(passage, answer)
        fkgl_source, fkgl_gen, fkgl_shift, syn_score = self.syntactic_score(passage, answer)

        final_score = round((lex_score + syn_score) / 2, 5)

        return {
            "source_zipf": zipf_source,
            "generated_zipf": zipf_gen,
            "zipf_shift": zipf_shift,
            "lexical_score": lex_score,
            "source_fkgl": fkgl_source,
            "generated_fkgl": fkgl_gen,
            "fkgl_shift": fkgl_shift,
            "syntactic_score": syn_score,
            "simplicity_score": final_score
        }

class LinguisticScore:
    """
    Comprehensive linguistic quality evaluation combining:
    - Fluency (GPT-2 loss)
    - Readability (FKGL + FRE)
    - Simplicity (Lexical + Syntactic)

    Final Scores:
    - Individual dimension scores
    - Combined score (average of all dimensions)
    - Delta score (generated - source)

    Score Interpretation Guide:

    Fluency (0-1):
    0.0-0.3: Much less fluent than source
    0.3-0.6: Similar fluency to source
    0.6-1.0: More fluent than source

    Readability (0-1):
    0.0-0.3: Much harder to read than source
    0.3-0.6: Similar readability to source
    0.6-1.0: Easier to read than source

    Simplicity (0-1):
    0.0-0.3: Much more complex than source
    0.3-0.6: Similar complexity to source
    0.6-1.0: Simpler than source

    Combined Score (0-1):
    Overall linguistic quality improvement
    """
    def __init__(self, device=None):
        self.fluency = FluencyScore(device=device)
        self.readability = ReadabilityScore()
        self.simplicity = SimplicityScore()

    def compute_all(self, passage: str, answer: str) -> Dict:
        """
        Compute all linguistic quality metrics

        Args:
            passage: Source text
            answer: Generated text to evaluate

        Returns:
            Dictionary containing all scores and metrics
        """
        if not passage or not answer:
            return {
                "fluency_score": 0,
                "fluency_loss": (0, 0),
                "readability_score": (0, 0),
                "simplicity_score": (0, 0),
                "combined_score": (0, 0),
                "delta_combined": 0
            }

        try:
            flu = self.fluency.score([passage], [answer])
            read = self.readability.compute(passage, answer)
            simp = self.simplicity.compute(passage, answer)

            flu_score = flu["scores"][0] if flu["scores"] else 0
            flu_loss_src = flu["sources_loss"][0] if flu["sources_loss"] else 0
            flu_loss_gen = flu["generateds_loss"][0] if flu["generateds_loss"] else 0

            read_score_src, read_score_gen = read["readability_score"]
            simp_score_src = round((simp["lexical_score"] + simp["syntactic_score"]) / 2, 5)
            simp_score_gen = simp["simplicity_score"]

            combined_src = round((1 / (1 + flu_loss_src) + read_score_src + simp_score_src) / 3, 5)
            combined_gen = round((1 / (1 + flu_loss_gen) + read_score_gen + simp_score_gen) / 3, 5)
            delta = round(combined_gen - combined_src, 5)

            return {
                "fluency_score": flu_score,
                "fluency_loss": (flu_loss_src, flu_loss_gen),
                "readability_score": (read_score_src, read_score_gen),
                "simplicity_score": (simp_score_src, simp_score_gen),
                "combined_score": (combined_src, combined_gen),
                "delta_combined": delta
            }
        except Exception as e:
            print(f"Error in compute_all: {str(e)}")
            return {
                "fluency_score": 0,
                "fluency_loss": (0, 0),
                "readability_score": (0, 0),
                "simplicity_score": (0, 0),
                "combined_score": (0, 0),
                "delta_combined": 0
            }

def validate_input_data(data: Union[Dict, List[Dict]]) -> List[Dict]:
    """Ensure input data is properly formatted"""
    if not data:
        return []

    if isinstance(data, dict):
        data = [data]

    valid_items = []
    for item in data:
        if not isinstance(item, dict):
            continue

        qid = item.get("QuestionID", f"item_{len(valid_items)}")
        passage = " ".join(item.get("RetrievedPassages", [])) if isinstance(item.get("RetrievedPassages", []), list) else ""
        answer = str(item.get("Answer", "")).strip()

        if passage and answer:
            valid_items.append({
                "QuestionID": qid,
                "RetrievedPassages": passage,
                "Answer": answer
            })

    return valid_items

def main():
    # Configuration
    input_json_file = "/content/drive/Othercomputers/MBZUAI/MBZUAI/ADGM-Project/SharedTask/TestSet/sample.json"
    output_folder_path = "/content/drive/Othercomputers/MBZUAI/MBZUAI/ADGM-Project/SharedTask/TestSet/"
    method_name = "linguistic_score_eval"
    team_name = "team_1"

    # Initialize
    start_time = time.time()
    print("Starting linguistic evaluation...")

    # Prepare output directory
    output_dir = os.path.join(output_folder_path, method_name, team_name)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output will be saved to: {output_dir}")

    # Load and validate input data
    try:
        with open(input_json_file, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        data = validate_input_data(raw_data)
        total_items = len(data)
        print(f"Loaded {total_items} valid items from input file")
    except Exception as e:
        print(f"Error loading input file: {str(e)}")
        return

    if not data:
        print("No valid data to process")
        return

    # Initialize scorer
    scorer = LinguisticScore()
    rows = []
    processed_count = 0
    error_count = 0

    # Initialize totals for averages
    metrics = {
        'flu': 0,
        'read_src': 0,
        'read_gen': 0,
        'simp_src': 0,
        'simp_gen': 0,
        'comb_src': 0,
        'comb_gen': 0,
        'delta': 0
    }

    # Process each item
    print("\nProcessing items:")
    for idx, item in enumerate(data, 1):
        try:
            print(f"  Processing item {idx}/{total_items}...", end='\r')

            result = scorer.compute_all(item["RetrievedPassages"], item["Answer"])

            row = {
                "QuestionID": item["QuestionID"],
                "FluencyScore": result["fluency_score"],
                "FluencyLoss_Source": result["fluency_loss"][0],
                "FluencyLoss_Generated": result["fluency_loss"][1],
                "ReadabilityScore_Source": result["readability_score"][0],
                "ReadabilityScore_Generated": result["readability_score"][1],
                "SimplicityScore_Source": result["simplicity_score"][0],
                "SimplicityScore_Generated": result["simplicity_score"][1],
                "CompositeLinguisticScore_Source": result["combined_score"][0],
                "CompositeLinguisticScore_Generated": result["combined_score"][1],
                "CompositeLinguisticScore_Delta": result["delta_combined"]
            }
            rows.append(row)

            # Update totals
            metrics['flu'] += result["fluency_score"]
            metrics['read_src'] += result["readability_score"][0]
            metrics['read_gen'] += result["readability_score"][1]
            metrics['simp_src'] += result["simplicity_score"][0]
            metrics['simp_gen'] += result["simplicity_score"][1]
            metrics['comb_src'] += result["combined_score"][0]
            metrics['comb_gen'] += result["combined_score"][1]
            metrics['delta'] += result["delta_combined"]

            processed_count += 1
        except Exception as e:
            error_count += 1
            print(f"\nError processing item {idx}: {str(e)}")

    # Calculate averages
    if processed_count > 0:
        avg_metrics = {k: v / processed_count for k, v in metrics.items()}
    else:
        avg_metrics = {k: 0 for k in metrics.keys()}

    # Write CSV results
    csv_file_path = os.path.join(output_dir, "linguistic_scores.csv")
    try:
        with open(csv_file_path, 'w', newline='', encoding='utf-8') as csvfile:
            if rows:
                fieldnames = rows[0].keys()
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)
        print(f"\nSuccessfully wrote {len(rows)} rows to {csv_file_path}")
    except Exception as e:
        print(f"\nError writing CSV file: {str(e)}")

    # Write minimal summary with only scores
    # Write minimal summary with concise score lines
    txt_file_path = os.path.join(output_dir, "linguistic_score_summary.txt")
    try:
        with open(txt_file_path, 'w', encoding='utf-8') as txtfile:
            txtfile.write("LINGUISTIC QUALITY EVALUATION SUMMARY\n")
            txtfile.write("=====================================\n\n")
            txtfile.write(f"Evaluation completed at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            txtfile.write(f"Total items processed: {processed_count}\n")
            txtfile.write(f"Items with errors: {error_count}\n")
            txtfile.write(f"Processing time: {time.time() - start_time:.2f} seconds\n\n")

            txtfile.write("AVERAGE SCORES\n")
            txtfile.write("--------------\n")
            txtfile.write(f"Fluency Score:       Source = {avg_metrics['comb_src']:.4f}, Generated = {avg_metrics['comb_gen']:.4f}, Delta = {avg_metrics['delta']:+.4f}\n")
            txtfile.write(f"Readability Score:   Source = {avg_metrics['read_src']:.4f}, Generated = {avg_metrics['read_gen']:.4f}, Delta = {avg_metrics['read_gen'] - avg_metrics['read_src']:+.4f}\n")
            txtfile.write(f"Simplicity Score:    Source = {avg_metrics['simp_src']:.4f}, Generated = {avg_metrics['simp_gen']:.4f}, Delta = {avg_metrics['simp_gen'] - avg_metrics['simp_src']:+.4f}\n")
            txtfile.write(f"\nFinal Linguistic Score: Source = {avg_metrics['comb_src']:.4f}, Generated = {avg_metrics['comb_gen']:.4f}, Delta = {avg_metrics['delta']:+.4f}\n")
    except Exception as e:
        print(f"Error writing summary file: {str(e)}")

    # Final report
    print("\nEvaluation Complete!")
    print(f"- Processed: {processed_count} items")
    print(f"- Errors: {error_count} items")
    print(f"- Time: {time.time() - start_time:.2f} seconds")
    print(f"\nFinal Linguistic Score: {avg_metrics['delta']:.4f}")

if __name__ == "__main__":
    main()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Starting linguistic evaluation...
Output will be saved to: /content/drive/Othercomputers/MBZUAI/MBZUAI/ADGM-Project/SharedTask/TestSet/linguistic_score_eval/team_1
Loaded 10 valid items from input file

Processing items:
  Processing item 10/10...
Successfully wrote 10 rows to /content/drive/Othercomputers/MBZUAI/MBZUAI/ADGM-Project/SharedTask/TestSet/linguistic_score_eval/team_1/linguistic_scores.csv

Evaluation Complete!
- Processed: 10 items
- Errors: 0 items
- Time: 1.27 seconds

Final Linguistic Score: -0.0339
